# 02 - Feature Engineering
## Construcao de features para o modelo de ranking de acoes

- Retornos em multiplos horizontes
- Indicadores de volatilidade
- Indicadores tecnicos (RSI, MACD, Bollinger, ATR)
- Features de volume
- Betas rolantes dos fatores Fama-French
- Rankings cross-sectional
- Variavel target (retorno futuro de 21 dias)

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df = pd.read_parquet("../data/raw/sp500_daily_prices.parquet")
print(f"Shape: {df.shape}")
print(f"Tickers: {df.index.get_level_values('ticker').nunique()}")
df.head()

Shape: (1943062, 6)
Tickers: 503


open       high        low      close  adj_close  \
date       ticker                                                          
2010-01-04 a       22.453505  22.625179  22.267525  22.389128  19.810986   
           aapl     7.622500   7.660714   7.585000   7.643214   6.412382   
           abt     26.000362  26.177889  25.870815  26.129908  18.207748   
           acgl     7.978889   8.022222   7.972222   7.994444   7.601905   
           acn     41.520000  42.200001  41.500000  42.070000  31.227369   

                        volume  
date       ticker               
2010-01-04 a         3815561.0  
           aapl    493729600.0  
           abt      10829095.0  
           acgl      4813200.0  
           acn       3650100.0

### 2.1 Retornos logaritmicos
 Retornos logaritmicos de 1, 5, 10, 21 e 63 dias (aproximadamente 1 dia, 1 semana, 2 semanas, 1 mes, 3 meses). Capturam momentum de curto e medio prazo. O modelo vai usar esses retornos passados como features pra prever retornos futuros. Retornos futuros de 21 dias (1 mes) vao ser o nosso target, ou seja, o que o modelo tenta prever.

In [6]:
horizons = [1, 5, 10, 21, 63]

for h in horizons:
    df[f"ret_{h}d"] = df.groupby("ticker")["close"].transform(
        lambda x: np.log(x / x.shift(h))
    )

print("Retornos calculados:")
print(df.filter(like="ret_").describe().round(4))

Retornos calculados:
             ret_1d        ret_5d       ret_10d       ret_21d       ret_63d
count  1.942559e+06  1.940547e+06  1.938032e+06  1.932499e+06  1.911373e+06
mean   4.000000e-04  2.200000e-03  4.400000e-03  9.200000e-03  2.790000e-02
std    2.030000e-02  4.450000e-02  6.170000e-02  8.890000e-02  1.475000e-01
min   -1.717900e+00 -1.718100e+00 -1.971300e+00 -2.053200e+00 -2.182900e+00
25%   -8.300000e-03 -1.810000e-02 -2.470000e-02 -3.440000e-02 -4.700000e-02
50%    7.000000e-04  3.400000e-03  6.400000e-03  1.260000e-02  3.440000e-02
75%    9.500000e-03  2.400000e-02  3.610000e-02  5.630000e-02  1.095000e-01
max    5.573000e-01  7.850000e-01  1.028500e+00  1.155400e+00  1.991300e+00


### 2.2 Variavel target
O target é o retorno log dos próximos 21 dias uteis (~1 mês).
Usamos shift negativo porque estamos olhando pra frente.

In [7]:
df["target_21d"] = df.groupby("ticker")["close"].transform(
    lambda x: np.log(x.shift(-21) / x)
)

print(f"Target nulo (ultimos 21 dias de cada ticker): {df['target_21d'].isna().sum()}")
print(f"Target preenchido: {df['target_21d'].notna().sum()}")

Target nulo (ultimos 21 dias de cada ticker): 10563
Target preenchido: 1932499


### 2.3 Volatilidade
Garman-Klass usa os quatro precos do dia (OHLC) para estimar volatilidade
com mais precisao que o desvio padrao simples do fechamento.
Tambem calculamos o desvio padrao rolante dos retornos diarios.

In [8]:
gk_var = (
    0.5 * (np.log(df["high"] / df["low"])) ** 2
    - (2 * np.log(2) - 1) * (np.log(df["close"] / df["open"])) ** 2
)
df["volatility_gk_21d"] = gk_var.groupby("ticker").transform(
    lambda x: x.rolling(21).mean()
)
df["volatility_gk_63d"] = gk_var.groupby("ticker").transform(
    lambda x: x.rolling(63).mean()
)

df["volatility_std_21d"] = df.groupby("ticker")["ret_1d"].transform(
    lambda x: x.rolling(21).std()
)
df["volatility_std_63d"] = df.groupby("ticker")["ret_1d"].transform(
    lambda x: x.rolling(63).std()
)

print("Volatilidade:")
print(df.filter(like="volatility").describe().round(6))

Volatilidade:
       volatility_gk_21d  volatility_gk_63d  volatility_std_21d  \
count       1.933002e+06       1.911876e+06        1.932499e+06   
mean        2.890000e-04       2.880000e-04        1.738900e-02   
std         4.890000e-04       3.830000e-04        1.055400e-02   
min         0.000000e+00       0.000000e+00        0.000000e+00   
25%         1.120000e-04       1.230000e-04        1.097100e-02   
50%         1.820000e-04       1.920000e-04        1.481800e-02   
75%         3.120000e-04       3.150000e-04        2.054800e-02   
max         2.951400e-02       1.232600e-02        3.769730e-01   

       volatility_std_63d  
count        1.911373e+06  
mean         1.797800e-02  
std          9.425000e-03  
min          0.000000e+00  
25%          1.209300e-02  
50%          1.564900e-02  
75%          2.092600e-02  
max          2.171980e-01  


### 2.4 Indicadores tecnicos
RSI: mede forca relativa do movimento de preco (0-100).
MACD: diferenca entre medias moveis exponenciais rapida e lenta.
Bollinger Band Width: distancia entre bandas superior e inferior.
ATR: amplitude media do movimento diario.

In [9]:
def calc_rsi(series, period):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def calc_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line, macd_line - signal_line


def calc_bollinger_width(series, period=20):
    sma = series.rolling(period).mean()
    std = series.rolling(period).std()
    upper = sma + 2 * std
    lower = sma - 2 * std
    return (upper - lower) / sma


def calc_atr(high, low, close, period=14):
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return tr.rolling(period).mean()


for ticker, group in df.groupby("ticker"):
    idx = group.index

    df.loc[idx, "rsi_14"] = calc_rsi(group["close"], 14)
    df.loc[idx, "rsi_21"] = calc_rsi(group["close"], 21)

    macd_line, signal_line, macd_hist = calc_macd(group["close"])
    df.loc[idx, "macd"] = macd_line
    df.loc[idx, "macd_signal"] = signal_line
    df.loc[idx, "macd_hist"] = macd_hist

    df.loc[idx, "bb_width"] = calc_bollinger_width(group["close"])
    df.loc[idx, "atr_14"] = calc_atr(group["high"], group["low"], group["close"])

print("Indicadores tecnicos:")
print(df[["rsi_14", "rsi_21", "macd_hist", "bb_width", "atr_14"]].describe().round(4))

Indicadores tecnicos:
             rsi_14        rsi_21     macd_hist      bb_width        atr_14
count  1.935844e+06  1.932628e+06  1.943062e+06  1.933505e+06  1.936523e+06
mean   5.289660e+01  5.277970e+01  2.000000e-03  1.182000e-01  2.685000e+00
std    1.690500e+01  1.394180e+01  1.432900e+00  8.640000e-02  6.211200e+00
min    0.000000e+00  0.000000e+00 -1.760427e+02  0.000000e+00  0.000000e+00
25%    4.090290e+01  4.292930e+01 -1.902000e-01  6.570000e-02  7.407000e-01
50%    5.306120e+01  5.285470e+01  4.200000e-03  9.610000e-02  1.363600e+00
75%    6.506550e+01  6.267880e+01  2.015000e-01  1.436000e-01  2.792900e+00
max    1.000000e+02  1.000000e+02  9.955440e+01  3.712000e+00  4.010593e+02


### 2.5 Features de volume
Dollar volume: preco x volume negociado (proxy de liquidez).
Volume ratio: volume atual vs media movel de 21 dias (detecta picos).
OBV: On Balance Volume, acumula volume em dias de alta e subtrai em dias de baixa.

In [10]:
df["dollar_volume"] = df["close"] * df["volume"]

df["volume_ratio_21d"] = df.groupby("ticker").apply(
    lambda g: g["volume"] / g["volume"].rolling(21).mean(),
    include_groups=False
).droplevel(0).sort_index()

def calc_obv(group):
    sign = np.sign(group["close"].diff())
    return (sign * group["volume"]).cumsum()

df["obv"] = df.groupby("ticker").apply(
    calc_obv, include_groups=False
).droplevel(0).sort_index()

df["obv_pct_change_21d"] = df.groupby("ticker")["obv"].transform(
    lambda x: x.pct_change(21)
)

print("Volume features:")
print(df[["dollar_volume", "volume_ratio_21d", "obv_pct_change_21d"]].describe().round(4))

Volume features:
       dollar_volume  volume_ratio_21d  obv_pct_change_21d
count   1.943062e+06      1.932700e+06        1.931996e+06
mean    3.843019e+08      1.010000e+00                 NaN
std     1.356716e+09      5.213000e-01                 NaN
min     0.000000e+00      0.000000e+00                -inf
25%     7.699325e+07      7.274000e-01       -4.370000e-02
50%     1.573519e+08      9.103000e-01        6.700000e-03
75%     3.324240e+08      1.159200e+00        6.190000e-02
max     1.543777e+11      2.100000e+01                 inf


### 2.6 Fatores Fama-French

Baixamos os 5 fatores Fama-French diretamente do site de Kenneth French (Dartmouth).
O arquivo vem compactado em ZIP com um CSV dentro.
Dividimos por 100 porque os valores originais estao em percentual.
Esses fatores serao usados na celula seguinte para calcular betas rolantes
de cada acao, medindo sua sensibilidade a fatores como mercado, tamanho e valor.

In [11]:
import zipfile
import io
import requests

url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    csv_name = z.namelist()[0]
    with z.open(csv_name) as f:
        ff_raw = pd.read_csv(f, skiprows=3)

ff_raw.columns = ff_raw.columns.str.strip()
ff_raw = ff_raw.rename(columns={ff_raw.columns[0]: "date"})
ff_raw["date"] = pd.to_datetime(ff_raw["date"], format="%Y%m%d", errors="coerce")
ff_factors = ff_raw.dropna(subset=["date"]).set_index("date")
ff_factors = ff_factors.apply(pd.to_numeric, errors="coerce") / 100

print(f"Fatores Fama-French: {ff_factors.shape}")
print(f"Periodo: {ff_factors.index.min()} ate {ff_factors.index.max()}")
ff_factors.head()

Fatores Fama-French: (15770, 6)
Periodo: 1963-07-01 00:00:00 ate 2026-02-27 00:00:00


,Mkt-RF,SMB,HML,RMW,CMA,RF
date,,,,,,
1963-07-01,-0.0067,0.0000,-0.0034,-0.0001,0.0016,0.0001
1963-07-02,0.0079,-0.0026,0.0026,-0.0007,-0.0020,0.0001
1963-07-03,0.0063,-0.0017,-0.0009,0.0018,-0.0034,0.0001
1963-07-05,0.0040,0.0008,-0.0027,0.0009,-0.0034,0.0001
1963-07-08,-0.0063,0.0004,-0.0018,-0.0029,0.0014,0.0001


In [15]:
factor_cols = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]
window = 252

daily_returns = df["ret_1d"].unstack("ticker")
daily_returns.index = pd.to_datetime(daily_returns.index)

ff_aligned = ff_factors.reindex(daily_returns.index)

print("Calculando betas rolantes ...")

beta_long = pd.DataFrame()

for factor in factor_cols:
    factor_series = ff_aligned[factor]
    beta_name = f"beta_{factor.lower().replace('-', '_')}"

    rolling_cov = daily_returns.rolling(window).cov(factor_series)
    rolling_var = factor_series.rolling(window).var()
    beta_wide = rolling_cov.div(rolling_var, axis=0)

    stacked = beta_wide.stack()
    stacked.name = beta_name
    stacked.index.names = ["date", "ticker"]

    if beta_long.empty:
        beta_long = stacked.to_frame()
    else:
        beta_long[beta_name] = stacked

df = df.join(beta_long, how="left")

print(f"Concluido. Shape do df: {df.shape}")
print(df.filter(like="beta_").describe().round(4))

Calculando betas rolantes ...


ValueError: columns overlap but no suffix specified: Index(['beta_mkt_rf', 'beta_smb', 'beta_hml', 'beta_rmw', 'beta_cma'], dtype='str')

### 2.7 Rankings cross-sectional
Para cada feature, calculamos o ranking percentil de cada acao em relacao
as demais no mesmo dia. Isso normaliza as features e permite comparacoes
justas entre acoes com escalas de preco diferentes.

In [16]:
features_to_rank = [
    "ret_1d", "ret_5d", "ret_21d", "ret_63d",
    "volatility_gk_21d", "volatility_std_21d",
    "rsi_14", "macd_hist", "bb_width", "atr_14",
    "volume_ratio_21d", "obv_pct_change_21d",
    "beta_mkt_rf", "beta_smb", "beta_hml"
]

for feat in features_to_rank:
    if feat in df.columns:
        df[f"{feat}_rank"] = df.groupby(level="date")[feat].rank(pct=True)

print(f"Features de ranking adicionadas: {len([c for c in df.columns if c.endswith('_rank')])}")
print(f"Shape final: {df.shape}")

Features de ranking adicionadas: 15
Shape final: (1943062, 47)


### 2.8 Filtro de liquidez
Mantemos apenas as 150 acoes mais liquidas em cada mes.
Acoes iliquidas geram sinais pouco confiaveis e sao caras pra operar.

In [18]:
df["year_month"] = df.index.get_level_values("date").to_period("M")

monthly_dollar_vol = df.groupby(["year_month", "ticker"])["dollar_volume"].mean()

top_150_per_month = (
    monthly_dollar_vol
    .groupby("year_month")
    .rank(ascending=False)
    .le(150)
)

liquid_pairs = set(
    (ym, tk)
    for (ym, tk), keep in top_150_per_month.items()
    if keep
)

df["is_liquid"] = [
    (ym, tk) in liquid_pairs
    for ym, tk in zip(df["year_month"], df.index.get_level_values("ticker"))
]

df_filtered = df[df["is_liquid"]].copy()
df_filtered = df_filtered.drop(columns=["is_liquid", "year_month"])

print(f"Antes do filtro: {df.index.get_level_values('ticker').nunique()} tickers")
print(f"Depois do filtro: {df_filtered.index.get_level_values('ticker').nunique()} tickers unicos")
print(f"Shape filtrado: {df_filtered.shape}")

Antes do filtro: 503 tickers
Depois do filtro: 406 tickers unicos
Shape filtrado: (615043, 47)


### 2.9 Limpeza e salvamento
Removemos linhas com NaN nas features principais (causados pelas janelas rolantes)
e salvamos o dataset processado.

In [19]:
core_features = [
    "ret_1d", "ret_5d", "ret_21d", "ret_63d",
    "volatility_gk_21d", "volatility_std_21d",
    "rsi_14", "macd_hist", "bb_width", "atr_14",
    "volume_ratio_21d",
    "beta_mkt_rf",
    "target_21d"
]

df_clean = df_filtered.dropna(subset=core_features)

print(f"Antes da limpeza: {df_filtered.shape}")
print(f"Depois da limpeza: {df_clean.shape}")
print(f"Periodo: {df_clean.index.get_level_values('date').min()} ate {df_clean.index.get_level_values('date').max()}")
print(f"Tickers: {df_clean.index.get_level_values('ticker').nunique()}")
print(f"\nColunas ({len(df_clean.columns)}):")
print(df_clean.columns.tolist())

Antes da limpeza: (615043, 47)
Depois da limpeza: (567503, 47)
Periodo: 2011-01-03 00:00:00 ate 2026-02-27 00:00:00
Tickers: 395

Colunas (47):
['open', 'high', 'low', 'close', 'adj_close', 'volume', 'ret_1d', 'ret_5d', 'ret_10d', 'ret_21d', 'ret_63d', 'target_21d', 'volatility_gk_21d', 'volatility_gk_63d', 'volatility_std_21d', 'volatility_std_63d', 'rsi_14', 'rsi_21', 'macd', 'macd_signal', 'macd_hist', 'bb_width', 'atr_14', 'dollar_volume', 'volume_ratio_21d', 'obv', 'obv_pct_change_21d', 'beta_mkt_rf', 'beta_smb', 'beta_hml', 'beta_rmw', 'beta_cma', 'ret_1d_rank', 'ret_5d_rank', 'ret_21d_rank', 'ret_63d_rank', 'volatility_gk_21d_rank', 'volatility_std_21d_rank', 'rsi_14_rank', 'macd_hist_rank', 'bb_width_rank', 'atr_14_rank', 'volume_ratio_21d_rank', 'obv_pct_change_21d_rank', 'beta_mkt_rf_rank', 'beta_smb_rank', 'beta_hml_rank']


In [20]:
df_clean.to_parquet("../data/processed/features.parquet")
print("Salvo em data/processed/features.parquet")
print(f"Tamanho: {df_clean.shape[0]:,} linhas x {df_clean.shape[1]} colunas")

Salvo em data/processed/features.parquet
Tamanho: 567,503 linhas x 47 colunas
